# T11G Compare Tasks


In [ ]:
from getpass import getpass
from pathlib import Path
import os
import subprocess
import sys

GITHUB_REPO = '2Kentaro1/wood-moisture-2d-cnn'
PROJECT_DIR = Path('/content/wood-moisture-2d-cnn')
DRIVE_OUTPUT_DIR = Path('/content/drive/MyDrive/wood-moisture-2d-cnn-outputs')
REPO_URL = f'https://github.com/{GITHUB_REPO}.git'

try:
    from google.colab import drive
    drive.mount('/content/drive')
    DRIVE_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    os.environ['OUTPUT_DIR'] = str(DRIVE_OUTPUT_DIR)
except Exception as exc:
    print(f'Google Drive mount skipped: {exc}')
    os.environ.setdefault('OUTPUT_DIR', 'outputs')

token = os.environ.get('GITHUB_TOKEN', '').strip()
if not token:
    token = getpass('GitHub token (empty for public repo): ').strip()
clone_url = REPO_URL if not token else f'https://{token}@github.com/{GITHUB_REPO}.git'

if PROJECT_DIR.exists() and (PROJECT_DIR / '.git').exists():
    os.chdir(PROJECT_DIR)
    try:
        if token:
            subprocess.run(['git', 'pull', clone_url, 'main'], check=True)
            subprocess.run(['git', 'remote', 'set-url', 'origin', REPO_URL], check=False)
        else:
            subprocess.run(['git', 'pull'], check=True)
    except subprocess.CalledProcessError as exc:
        print(f'git pull skipped/failed: {exc}')
else:
    PROJECT_DIR.parent.mkdir(parents=True, exist_ok=True)
    if PROJECT_DIR.exists() and not (PROJECT_DIR / '.git').exists():
        raise RuntimeError(f'{PROJECT_DIR} exists but is not a git repository. Rename/remove it or set PROJECT_DIR.')
    subprocess.run(['git', 'clone', clone_url, str(PROJECT_DIR)], check=True)
    os.chdir(PROJECT_DIR)
    subprocess.run(['git', 'remote', 'set-url', 'origin', REPO_URL], check=False)

os.environ['PROJECT_ROOT'] = str(PROJECT_DIR)
if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))
Path(os.environ['OUTPUT_DIR']).mkdir(parents=True, exist_ok=True)

print(f'PROJECT_ROOT={os.environ["PROJECT_ROOT"]}')
print(f'OUTPUT_DIR={os.environ["OUTPUT_DIR"]}')
print('src exists:', (PROJECT_DIR / 'src').exists())
print('train exists:', (PROJECT_DIR / 'data' / 'train.csv').exists())
print('test exists:', (PROJECT_DIR / 'data' / 'test.csv').exists())

subprocess.run([sys.executable, '-m', 'pip', 'install', '-r', 'requirements.txt'], check=True)


In [ ]:
import os
from pathlib import Path
import numpy as np
from src.data.load_data import load_train_test
from src.interpret.visualization import plot_task_comparison_heatmap, plot_difference_map

output_dir = Path(os.environ.get('OUTPUT_DIR', 'outputs'))
train_frame, _ = load_train_test('.')
task_importance = {}
for task in ['mc', 'species', 'woodtype', 'wood_structure', 'index_norm', 'mc_norm']:
    path = output_dir / 'heatmaps' / f'{task}_importance.npy'
    if path.exists():
        task_importance[task] = np.load(path)

if task_importance:
    plot_task_comparison_heatmap(task_importance, train_frame.wavelengths, output_dir / 'figures' / 'task_comparison_heatmap.png', overwrite=True)
if {'mc', 'species'} <= set(task_importance):
    plot_difference_map(task_importance['mc'], task_importance['species'], train_frame.wavelengths, 'MC - species', output_dir / 'figures' / 'diff_mc_species.png', overwrite=True)
if {'index_norm', 'mc_norm'} <= set(task_importance):
    plot_difference_map(task_importance['index_norm'], task_importance['mc_norm'], train_frame.wavelengths, 'index_norm - mc_norm', output_dir / 'figures' / 'diff_index_mc_norm.png', overwrite=True)
if {'species', 'woodtype'} <= set(task_importance):
    plot_difference_map(task_importance['species'], task_importance['woodtype'], train_frame.wavelengths, 'species - woodtype', output_dir / 'figures' / 'diff_species_woodtype.png', overwrite=True)
if {'species', 'wood_structure'} <= set(task_importance):
    plot_difference_map(task_importance['species'], task_importance['wood_structure'], train_frame.wavelengths, 'species - wood_structure', output_dir / 'figures' / 'diff_species_wood_structure.png', overwrite=True)
